In [1]:
import os
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [7]:
load_dotenv()
db_uri = os.getenv("DATABASE_URI")
sql_query = "SELECT * FROM btc_usdt_hourly ORDER BY open_time ASC;"

In [8]:
conn = psycopg2.connect(db_uri)
df = pd.read_sql_query(sql_query,conn)
conn.close()

C:\Users\vedat\AppData\Local\Temp\ipykernel_12672\4030805866.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql_query,conn)


In [9]:
df.head()

,open_time,open_price,high_price,low_price,close_price,volume
0,2020-01-01 03:00:00,7195.24,7196.25,7175.46,7177.02,511.814901
1,2020-01-01 04:00:00,7176.47,7230.00,7175.71,7216.27,883.052603
2,2020-01-01 05:00:00,7215.52,7244.87,7211.41,7242.85,655.156809
3,2020-01-01 06:00:00,7242.66,7245.00,7220.00,7225.01,783.724867
4,2020-01-01 07:00:00,7225.00,7230.00,7215.03,7217.27,467.812578


In [10]:
features = ['close_price', 'high_price', 'low_price', 'volume']
data = df[features].values

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)
print(scaled_data[:5])

[[0.0249948  0.02127032 0.0279354  0.00373023]
 [0.02531684 0.02154789 0.02793746 0.00643591]
 [0.02553492 0.02167019 0.02823136 0.00477495]
 [0.02538855 0.02167126 0.02830207 0.00571198]
 [0.02532504 0.02154789 0.02826116 0.00340953]]


In [19]:
look_back = 60

X_train = []
y_train = []

for i in range(look_back, len(scaled_data)):
    X_train.append(scaled_data[i-look_back:i, 0:4]) 
    y_train.append(scaled_data[i, 0])
X_train, y_train = np.array(X_train), np.array(y_train)

print(X_train.shape)
print(y_train.shape)

(50753, 60, 4)
(50753,)


In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam

model_V3 = Sequential()

model_V3.add(LSTM(units=100, return_sequences=True, 
                  input_shape=(X_train_multi.shape[1], X_train_multi.shape[2])))
model_V3.add(Dropout(0.2))

model_V3.add(LSTM(units=100, return_sequences=False))
model_V3.add(Dropout(0.2))

model_V3.add(Dense(units=1))

optimizer = Adam(learning_rate=0.0005)
model_V3.compile(optimizer=optimizer, loss='mean_squared_error')

In [28]:
model_V3.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                       │ (None, 60, 100)             │          42,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 60, 100)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_11 (LSTM)                       │ (None, 100)                 │          80,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 100)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 122,501 (478.52 KB)

 Trainable params: 122,501 (478.52 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history_V3 = model_V3.fit(
    X_train_multi, 
    y_train_multi,
    epochs=20, 
    batch_size=32
)

Epoch 1/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 29ms/step - loss: 2.2645e-04
Epoch 2/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.2502e-04
Epoch 3/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - loss: 2.2199e-04
Epoch 4/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.2236e-04
Epoch 5/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.2155e-04
Epoch 6/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - loss: 2.1807e-04
Epoch 7/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.2046e-04
Epoch 8/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.1986e-04
Epoch 9/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - loss: 2.1731e-04
Epoch 10/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.2329e-04
Epoch 11/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 46s 29ms/step - loss: 2.1567e-04
Epoch 12/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.0960e-04
Epoch 13/20
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 2.1548e-04
Epoch 14/20
1587/1587

In [33]:
from tensorflow.keras.optimizers import Adam
optimizer_fine_tune = Adam(learning_rate=0.00005)
model_V3.compile(optimizer=optimizer_fine_tune, loss='mean_squared_error')

history_V3_fine_tune = model_V3.fit(
    X_train_multi, 
    y_train_multi,
    epochs=10, 
    batch_size=32
)

Epoch 1/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 48s 29ms/step - loss: 1.8126e-04
Epoch 2/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 29ms/step - loss: 1.8104e-04
Epoch 3/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 1.8349e-04
Epoch 4/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 1.8255e-04
Epoch 5/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - loss: 1.8221e-04
Epoch 6/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - loss: 1.8117e-04
Epoch 7/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 1.8610e-04
Epoch 8/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 1.8167e-04
Epoch 9/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 29ms/step - loss: 1.8258e-04
Epoch 10/10
1587/1587 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 1.8282e-04


In [34]:
model_V3.save('finsmart_model_v1.keras')